# Termometro: prova di velocità e precisione su Kaggle
Impostazioni: acceleratore **GPU T4 x2**, **Internet attivo**. Dati in ingresso: il dataset privato con `termometro.py`, `countdown.py`, `velocita.py`. Nessun dato dell'esperimento.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers==5.5.4 peft==0.18.0
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__)

In [ ]:
# modello base, lo stesso di casa (si controlla l'impronta sha256 del file dei pesi)
import hashlib
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', local_dir='/kaggle/working/qwen')
h = hashlib.sha256(open('/kaggle/working/qwen/model.safetensors', 'rb').read()).hexdigest()
print(h, 'UGUALE a casa' if h.startswith('dd924a11') and h.endswith('c6d3ee') else 'DIVERSO da casa')

In [ ]:
import glob, os
D = os.path.dirname(glob.glob('/kaggle/input/**/velocita.py', recursive=True)[0])
print(D, os.listdir(D))

In [ ]:
# precisione piena (fp32), GPU 0, LoRA nuovo (non addestrato): solo velocità e precisione
!TERMO_FP32=1 CUDA_VISIBLE_DEVICES=0 python {D}/velocita.py {D} /kaggle/working/qwen none /kaggle/working/velocita_t4_fp32.json

In [ ]:
# bf16 come a casa, per vedere se la T4 la regge e quanto costa (se dà errore va bene: lo annotiamo)
!CUDA_VISIBLE_DEVICES=0 python {D}/velocita.py {D} /kaggle/working/qwen none /kaggle/working/velocita_t4_bf16.json || echo 'bf16 non funziona su T4'

In [ ]:
for f in sorted(glob.glob('/kaggle/working/velocita_*.json')):
    print(f, open(f).read())